# Deepfake Detection — Training & Inference Notebook

In [1]:
%pip install -q torch torchvision pandas pillow scikit-learn tqdm peft

Note: you may need to restart the kernel to use updated packages.


In [2]:
import copy
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights, swin_t, Swin_T_Weights
from peft import LoraConfig, get_peft_model
import peft.import_utils
import peft.tuners.lora.torchao
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import gc
import torch
import os

gc.collect()
torch.cuda.empty_cache()

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
class DeepfakeDataset(Dataset):
    def __init__(self, dir, isTrain, csv_path=None, transform=None, two_views=False):
        self.dir = dir
        self.isTrain = isTrain
        self.transform = transform
        self.two_views = two_views  # return two independently augmented views, stacked as [2, C, H, W]

        split_dir = os.path.join(dir, "train" if isTrain else "test")
        self.img_dir = split_dir if os.path.isdir(split_dir) else dir

        if csv_path is None:
            csv_path = "../ground_truth.csv" if isTrain else "../sample_submission.csv"
        full_csv_path = os.path.join(dir, csv_path)

        if os.path.exists(full_csv_path):
            self.labels = pd.read_csv(full_csv_path)
        else:
            image_ids = sorted(
                os.path.splitext(f)[0]
                for f in os.listdir(self.img_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            )
            self.labels = pd.DataFrame({"image_id": image_ids, "label": 0})

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        row = self.labels.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        image = Image.open(img_path).convert("RGB")
        label = torch.tensor(row["label"], dtype=torch.long)

        if self.transform is not None:
            if self.two_views:
                image = torch.stack([self.transform(image), self.transform(image)])
            else:
                image = self.transform(image)

        return image, label

    @staticmethod
    def load_dataloader(dataset, batch_size, val_split=0.0, num_workers=0, shuffle=True, seed=42, val_transform=None):
        if val_split and val_split > 0:
            val_size = int(len(dataset) * val_split)
            train_size = len(dataset) - val_size
            generator = torch.Generator().manual_seed(seed)
            train_subset, val_subset = random_split(dataset, [train_size, val_size], generator=generator)
            if val_transform is not None:
                # Validate on un-augmented images: same indices, eval transform
                val_dataset = copy.copy(dataset)
                val_dataset.transform = val_transform
                val_dataset.two_views = False
                val_subset = Subset(val_dataset, val_subset.indices)
            train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)
            val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
            return train_loader, val_loader

        return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers)

In [4]:
BACKBONES = {
    "efficientnet_b4": (efficientnet_b4, EfficientNet_B4_Weights.IMAGENET1K_V1),
    "convnext_tiny": (convnext_tiny, ConvNeXt_Tiny_Weights.IMAGENET1K_V1),
    "swin_t": (swin_t, Swin_T_Weights.IMAGENET1K_V1),
}


class DeepfakeDetectionModel(nn.Module):
    def __init__(self, arch="efficientnet_b4", pretrained=True, dropout=0.4, num_classes=1, freeze_backbone=False, use_lora=True, lora_r=8, lora_alpha=16, lora_dropout=0.1, proj_dim=0):
        super().__init__()
        self.arch = arch
        self.use_lora = use_lora
        build, weights = BACKBONES[arch]
        self.backbone = build(weights=weights if pretrained else None)

        # Replace the ImageNet classifier with a single-logit head
        if arch == "efficientnet_b4":
            in_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
            head_name = "classifier"
        elif arch == "convnext_tiny":  # classifier = [LayerNorm2d, Flatten, Linear]
            in_features = self.backbone.classifier[2].in_features
            self.backbone.classifier[2] = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
            head_name = "classifier"
        else:  # swin_t
            in_features = self.backbone.head.in_features
            self.backbone.head = nn.Sequential(nn.Dropout(p=dropout), nn.Linear(in_features, num_classes))
            head_name = "head"

        if use_lora:
            # EfficientNet: LoRA on the 1x1 convolutions. ConvNeXt / Swin: their pointwise and attention layers are nn.Linear.
            if arch == "efficientnet_b4":
                is_target = lambda m: isinstance(m, nn.Conv2d) and m.kernel_size == (1, 1)
            else:
                is_target = lambda m: isinstance(m, nn.Linear)
            target_names = [
                re.escape(f"features.{name}")
                for name, m in self.backbone.features.named_modules()
                if is_target(m)
            ]
            target_modules = f"^({'|'.join(target_names)})$"
            lora_config = LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                target_modules=target_modules,
                modules_to_save=[head_name],
            )
            self.backbone = get_peft_model(self.backbone, lora_config)
            self.backbone.print_trainable_parameters()
        elif freeze_backbone:
            for param in self.backbone.features.parameters():
                param.requires_grad = False

        # Projection head for contrastive training only. Not part of the exported / inference model.
        if proj_dim > 0:
            self.proj = nn.Sequential(
                nn.Linear(in_features, 512),
                nn.ReLU(inplace=True),
                nn.Linear(512, proj_dim),
            )

    def forward(self, img, return_embedding=False):
        if not return_embedding:
            return self.backbone(img)
        net = self.backbone
        if self.arch == "efficientnet_b4":
            feat = net.avgpool(net.features(img)).flatten(1)
            logits = net.classifier(feat)
        elif self.arch == "convnext_tiny":
            pooled = net.avgpool(net.features(img))
            feat = pooled.flatten(1)
            logits = net.classifier(pooled)
        else:  # swin_t
            feat = net.flatten(net.avgpool(net.permute(net.norm(net.features(img)))))
            logits = net.head(feat)
        return logits, nn.functional.normalize(self.proj(feat), dim=1)

    def save(self, path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        torch.save(self.state_dict(), path)

    def load(self, path, map_location=None):
        self.load_state_dict(torch.load(path, map_location=map_location, weights_only=True))

    def export_merged(self, path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        if self.use_lora:
            merged_backbone = copy.deepcopy(self.backbone).merge_and_unload()
            state_dict = {f"backbone.{k}": v for k, v in merged_backbone.state_dict().items()}
        else:
            state_dict = {k: v for k, v in self.state_dict().items() if not k.startswith("proj.")}
        torch.save(state_dict, path)

In [5]:
class DeepfakeDetectionPipeline:
    def __init__(self, dir, lr=1e-4, weight_decay=1e-4, label_smoothing=0.05,
                 consistency_weight=0.0, supcon_weight=0.0, temperature=0.1, **model_kwargs):
        self.dir = dir
        self.train_dir = os.path.join(dir, "train")
        self.test_dir = os.path.join(dir, "test")
        self.train_csv = os.path.join(dir, "ground_truth.csv")
        self.test_csv = os.path.join(dir, "sample_submission.csv")

        self.label_smoothing = label_smoothing
        self.consistency_weight = consistency_weight
        self.supcon_weight = supcon_weight
        self.temperature = temperature
        if supcon_weight > 0:
            model_kwargs.setdefault("proj_dim", 128)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = DeepfakeDetectionModel(**model_kwargs).to(self.device)

        trainable_params = filter(lambda p: p.requires_grad, self.model.parameters())
        self.optimiser = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
        self.criterion = nn.BCEWithLogitsLoss()

    def _compute_loss_and_probs(self, logits, labels=None, smooth=False):
        if logits.ndim == 2 and logits.shape[1] == 2:
            probs = torch.softmax(logits, dim=1)[:, 1]
            if labels is None:
                return None, probs
            smoothing = self.label_smoothing if smooth else 0.0
            loss = nn.functional.cross_entropy(logits, labels.long(), label_smoothing=smoothing)
            return loss, probs

        logits = logits.squeeze(-1)
        probs = torch.sigmoid(logits)
        if labels is None:
            return None, probs

        targets = labels.float()
        if smooth and self.label_smoothing > 0:
            targets = targets * (1.0 - self.label_smoothing) + 0.5 * self.label_smoothing
        loss = self.criterion(logits, targets)
        return loss, probs

    def _supcon_loss(self, emb, labels):
        # Supervised contrastive loss (Khosla et al. 2020) on L2-normalised embeddings:
        # pull together samples with the same label (incl. the other view of the same image), push apart the rest.
        sim = emb @ emb.T / self.temperature
        eye = torch.eye(len(emb), dtype=torch.bool, device=emb.device)
        sim = sim.masked_fill(eye, float("-inf"))
        pos = (labels[:, None] == labels[None, :]) & ~eye
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)
        return -(log_prob.masked_fill(~pos, 0.0).sum(1) / pos.sum(1).clamp(min=1)).mean()

    def evaluate(self, val_loader):
        self.model.eval()
        running_loss = 0.0
        all_probs = []
        all_labels = []

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validating", leave=False):
                images = images.to(self.device)
                labels = labels.to(self.device)

                logits = self.model(images)
                loss, probs = self._compute_loss_and_probs(logits, labels, smooth=False)

                running_loss += loss.item() * images.size(0)
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss = running_loss / len(val_loader.dataset)
        preds = [1 if p >= 0.5 else 0 for p in all_probs]
        val_auc = roc_auc_score(all_labels, all_probs)
        val_f1 = f1_score(all_labels, preds)
        val_acc = accuracy_score(all_labels, preds)
        return val_loss, val_auc, val_f1, val_acc

    def fit(self, train_loader, val_loader=None, epochs=10, save_path="./models/best_model.pt"):
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimiser, T_max=epochs)
        scaler = torch.amp.GradScaler("cuda", enabled=(self.device.type == "cuda"))
        best_val_auc = 0.0

        for epoch in range(1, epochs + 1):
            self.model.train()
            running_loss = 0.0

            pbar = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{epochs:02d}] Train", leave=False)
            for images, labels in pbar:
                images = images.to(self.device)
                labels = labels.to(self.device)

                batch_size = labels.size(0)

                # Two augmented views per image: [B, 2, C, H, W] -> [2B, C, H, W] (views of one image are adjacent)
                two_views = images.ndim == 5
                if two_views:
                    images = images.flatten(0, 1)
                    labels = labels.repeat_interleave(2)

                self.optimiser.zero_grad()
                with torch.amp.autocast(device_type=self.device.type, enabled=(self.device.type == "cuda")):
                    if self.supcon_weight > 0:
                        logits, emb = self.model(images, return_embedding=True)
                    else:
                        logits = self.model(images)
                    loss, probs = self._compute_loss_and_probs(logits, labels, smooth=True)
                    if two_views and self.consistency_weight > 0:
                        # Both views of an image should get the same fake-probability, whatever the degradation
                        p = probs.float().view(-1, 2)
                        loss = loss + self.consistency_weight * nn.functional.mse_loss(p[:, 0], p[:, 1])
                    if self.supcon_weight > 0:
                        loss = loss + self.supcon_weight * self._supcon_loss(emb.float(), labels)

                scaler.scale(loss).backward()
                scaler.step(self.optimiser)
                scaler.update()

                running_loss += loss.item() * batch_size
                pbar.set_postfix(loss=f"{loss.item():.4f}")

            scheduler.step()
            train_loss = running_loss / len(train_loader.dataset)

            if val_loader is not None:
                val_loss, val_auc, val_f1, val_acc = self.evaluate(val_loader)
                print(
                    f"Epoch [{epoch:02d}/{epochs:02d}] "
                    f"Train Loss: {train_loss:.4f} | "
                    f"Val Loss: {val_loss:.4f} | "
                    f"Val AUC: {val_auc:.4f} | "
                    f"Val F1: {val_f1:.4f} | "
                    f"Val Acc: {val_acc:.4f}"
                )
                if save_path and val_auc > best_val_auc:
                    best_val_auc = val_auc
                    self.save(save_path)
                    print(f"  -> Saved best model to {save_path} (Val AUC: {best_val_auc:.4f})")
            else:
                print(f"Epoch [{epoch:02d}/{epochs:02d}] Train Loss: {train_loss:.4f}")
                if save_path:
                    self.save(save_path)

    def predict(self, data_loader, threshold=0.5):
        self.model.eval()
        all_probs = []

        with torch.no_grad():
            for batch in tqdm(data_loader, desc="Predicting"):
                images = batch[0] if isinstance(batch, (tuple, list)) else batch
                images = images.to(self.device)
                logits = self.model(images)
                _, probs = self._compute_loss_and_probs(logits)
                all_probs.extend(probs.cpu().numpy())

        preds = [1 if p >= threshold else 0 for p in all_probs]
        return preds, all_probs

    def save(self, path):
        self.model.save(path)

    def load(self, path):
        self.model.load(path)

## Data

In [6]:
TRAIN_DIR = "/kaggle/input/datasets/humanoid2005/deepfakeml-hackathon/dataset/train"
TEST_DIR = "/kaggle/input/datasets/humanoid2005/deepfakeml-hackathon/dataset/test"
BATCH_SIZE = 32
VAL_SPLIT = 0.2
EPOCHS = 20
MODEL_DIR = "/kaggle/working/models"
OUTPUT_CSV = "/kaggle/working/predictions.csv"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05)
    ], p=0.5),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))
    ], p=0.2),
    # Degradations present in the test set but rare in train: pixelation, greyscale, posterise, false colour, block occlusion
    transforms.RandomApply([transforms.RandomChoice([
        transforms.Compose([transforms.Resize(s), transforms.Resize(224, interpolation=transforms.InterpolationMode.NEAREST)])
        for s in (32, 48, 64)
    ])], p=0.15),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPosterize(bits=2, p=0.1),
    transforms.RandomSolarize(threshold=128, p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.1), value="random"),
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = DeepfakeDataset(TRAIN_DIR, isTrain=True, transform=train_transforms, two_views=True)
test_dataset = DeepfakeDataset(TEST_DIR, isTrain=False, transform=eval_transforms)

# Same seed => every model below is trained and validated on the same split, so their validation predictions line up
train_loader, val_loader = DeepfakeDataset.load_dataloader(
    train_dataset, batch_size=BATCH_SIZE, val_split=VAL_SPLIT, val_transform=eval_transforms
)
test_loader = DeepfakeDataset.load_dataloader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

# Shared by all three models: LoRA fine-tuning, BCE + consistency (1.0) + supervised contrastive (0.1)
TRAIN_KWARGS = dict(lr=3e-4, use_lora=True, consistency_weight=1.0, supcon_weight=0.1)


def train_and_export(arch):
    pipeline = DeepfakeDetectionPipeline(os.path.dirname(TRAIN_DIR), arch=arch, **TRAIN_KWARGS)
    print(f"Training {arch} on {pipeline.device}")
    best_path = f"{MODEL_DIR}/{arch}_best.pt"
    pipeline.fit(train_loader, val_loader, epochs=EPOCHS, save_path=best_path)
    pipeline.load(best_path)
    pipeline.model.export_merged(f"{MODEL_DIR}/{arch}_weights.pt")  # plain backbone + head, LoRA merged in
    print(f"Exported {MODEL_DIR}/{arch}_weights.pt")
    del pipeline
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 1. Train EfficientNet-B4

In [7]:
train_and_export("efficientnet_b4")

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 176MB/s]


trainable params: 1,007,489 || all params: 18,557,898 || trainable%: 5.4289
Training efficientnet_b4 on cuda


Epoch [01/20] Train Loss: 1.0789 | Val Loss: 0.5605 | Val AUC: 0.7958 | Val F1: 0.7337 | Val Acc: 0.7293
  -> Saved best model to /kaggle/working/models/efficientnet_b4_best.pt (Val AUC: 0.7958)


Epoch [02/20] Train Loss: 0.9869 | Val Loss: 0.4740 | Val AUC: 0.8633 | Val F1: 0.8093 | Val Acc: 0.8064
  -> Saved best model to /kaggle/working/models/efficientnet_b4_best.pt (Val AUC: 0.8633)


Epoch [03/20] Train Loss: 0.9384 | Val Loss: 0.4418 | Val AUC: 0.8805 | Val F1: 0.8264 | Val Acc: 0.8293
  -> Saved best model to /kaggle/working/models/efficientnet_b4_best.pt (Val AUC: 0.8805)


... (13 outputs left out)

Epoch [17/20] Train Loss: 0.7525 | Val Loss: 0.3989 | Val AUC: 0.8959 | Val F1: 0.8426 | Val Acc: 0.8450


Epoch [18/20] Train Loss: 0.7453 | Val Loss: 0.4002 | Val AUC: 0.8966 | Val F1: 0.8346 | Val Acc: 0.8386
  -> Saved best model to /kaggle/working/models/efficientnet_b4_best.pt (Val AUC: 0.8966)


Epoch [19/20] Train Loss: 0.7480 | Val Loss: 0.4002 | Val AUC: 0.8955 | Val F1: 0.8354 | Val Acc: 0.8379


Epoch [20/20] Train Loss: 0.7493 | Val Loss: 0.4024 | Val AUC: 0.8947 | Val F1: 0.8339 | Val Acc: 0.8379
Exported /kaggle/working/models/efficientnet_b4_weights.pt


## 2. Train ConvNeXt-Tiny

In [8]:
train_and_export("convnext_tiny")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 205MB/s]  


trainable params: 532,225 || all params: 28,353,122 || trainable%: 1.8771
Training convnext_tiny on cuda


Epoch [01/20] Train Loss: 1.0527 | Val Loss: 0.5238 | Val AUC: 0.8434 | Val F1: 0.7735 | Val Acc: 0.7407
  -> Saved best model to /kaggle/working/models/convnext_tiny_best.pt (Val AUC: 0.8434)


Epoch [02/20] Train Loss: 0.9209 | Val Loss: 0.4278 | Val AUC: 0.8840 | Val F1: 0.8314 | Val Acc: 0.8343
  -> Saved best model to /kaggle/working/models/convnext_tiny_best.pt (Val AUC: 0.8840)


Epoch [03/20] Train Loss: 0.8735 | Val Loss: 0.4011 | Val AUC: 0.8962 | Val F1: 0.8426 | Val Acc: 0.8471
  -> Saved best model to /kaggle/working/models/convnext_tiny_best.pt (Val AUC: 0.8962)


... (13 outputs left out)

Epoch [17/20] Train Loss: 0.6504 | Val Loss: 0.4501 | Val AUC: 0.8869 | Val F1: 0.8451 | Val Acc: 0.8457


Epoch [18/20] Train Loss: 0.6403 | Val Loss: 0.4542 | Val AUC: 0.8838 | Val F1: 0.8333 | Val Acc: 0.8371


Epoch [19/20] Train Loss: 0.6355 | Val Loss: 0.4536 | Val AUC: 0.8853 | Val F1: 0.8388 | Val Acc: 0.8407


Epoch [20/20] Train Loss: 0.6341 | Val Loss: 0.4542 | Val AUC: 0.8852 | Val F1: 0.8388 | Val Acc: 0.8407
Exported /kaggle/working/models/convnext_tiny_weights.pt


## 3. Train Swin-Tiny

In [9]:
train_and_export("swin_t")

Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 216MB/s] 


trainable params: 598,273 || all params: 28,118,396 || trainable%: 2.1277
Training swin_t on cuda


Epoch [01/20] Train Loss: 1.0724 | Val Loss: 0.5301 | Val AUC: 0.8224 | Val F1: 0.7673 | Val Acc: 0.7436
  -> Saved best model to /kaggle/working/models/swin_t_best.pt (Val AUC: 0.8224)


Epoch [02/20] Train Loss: 0.9837 | Val Loss: 0.4643 | Val AUC: 0.8672 | Val F1: 0.8096 | Val Acc: 0.8079
  -> Saved best model to /kaggle/working/models/swin_t_best.pt (Val AUC: 0.8672)


Epoch [03/20] Train Loss: 0.9348 | Val Loss: 0.4661 | Val AUC: 0.8804 | Val F1: 0.7772 | Val Acc: 0.7993
  -> Saved best model to /kaggle/working/models/swin_t_best.pt (Val AUC: 0.8804)


... (13 outputs left out)

Epoch [17/20] Train Loss: 0.7542 | Val Loss: 0.3927 | Val AUC: 0.9019 | Val F1: 0.8451 | Val Acc: 0.8500


Epoch [18/20] Train Loss: 0.7555 | Val Loss: 0.3937 | Val AUC: 0.9014 | Val F1: 0.8423 | Val Acc: 0.8479


Epoch [19/20] Train Loss: 0.7533 | Val Loss: 0.3896 | Val AUC: 0.9018 | Val F1: 0.8500 | Val Acc: 0.8536


Epoch [20/20] Train Loss: 0.7522 | Val Loss: 0.3901 | Val AUC: 0.9017 | Val F1: 0.8500 | Val Acc: 0.8536
Exported /kaggle/working/models/swin_t_weights.pt


## Ensemble inference
Loads each exported model from disk (so this section also works in a fresh session), scores every model and every combination on the validation split, then averages the chosen models' probabilities on the test set.

In [10]:
import itertools

ARCHS = ["efficientnet_b4", "convnext_tiny", "swin_t"]
ENSEMBLE = ARCHS  # models averaged for predictions.csv; edit after looking at the table below
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_exported(arch):
    model = DeepfakeDetectionModel(arch=arch, pretrained=False, use_lora=False)
    model.load_state_dict(torch.load(f"{MODEL_DIR}/{arch}_weights.pt", map_location="cpu", weights_only=True))
    return model.to(device).eval()


@torch.no_grad()
def predict_probs(model, loader):
    out = []
    for images, _ in tqdm(loader, desc="Predicting", leave=False):
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(images.to(device))
        out.append(torch.sigmoid(logits.float().squeeze(-1)).cpu().numpy())
    return np.concatenate(out)


models = {arch: load_exported(arch) for arch in ARCHS}
val_labels = train_dataset.labels["label"].values[np.asarray(val_loader.dataset.indices)]
val_probs = {arch: predict_probs(m, val_loader) for arch, m in models.items()}

# Validation metrics for every single model and every combination (probability average)
rows = []
for k in range(1, len(ARCHS) + 1):
    for combo in itertools.combinations(ARCHS, k):
        p = np.mean([val_probs[a] for a in combo], axis=0)
        rows.append({
            "models": " + ".join(combo),
            "val_auc": roc_auc_score(val_labels, p),
            "val_f1": f1_score(val_labels, p >= 0.5),
            "val_acc": accuracy_score(val_labels, p >= 0.5),
        })
display(pd.DataFrame(rows).sort_values("val_auc", ascending=False).round(4))

# How similar are the models' predictions? Close to 1 = redundant, lower = more to gain from averaging
display(pd.DataFrame(np.corrcoef([val_probs[a] for a in ARCHS]), index=ARCHS, columns=ARCHS).round(3))

,models,val_auc,val_f1,val_acc
6,efficientnet_b4 + convnext_tiny + swin_t,0.9090,0.8559,0.8586
5,convnext_tiny + swin_t,0.9081,0.8598,0.8621
4,efficientnet_b4 + swin_t,0.9066,0.8604,0.8621
3,efficientnet_b4 + convnext_tiny,0.9060,0.8489,0.8536
2,swin_t,0.9043,0.8573,0.8571
1,convnext_tiny,0.9026,0.8418,0.8464
0,efficientnet_b4,0.8966,0.8346,0.8386


,efficientnet_b4,convnext_tiny,swin_t
efficientnet_b4,1.000,0.914,0.911
convnext_tiny,0.914,1.000,0.931
swin_t,0.911,0.931,1.000


In [11]:
# Test-set predictions from the ensemble
test_probs = np.mean([predict_probs(models[a], test_loader) for a in ENSEMBLE], axis=0)
submission = pd.DataFrame({
    "image_id": test_dataset.labels["image_id"].tolist(),
    "label": (test_probs >= 0.5).astype(int),
    "confidence": test_probs,
})
submission.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(submission)} predictions from {ENSEMBLE} to {OUTPUT_CSV}")
submission.head()

Saved 3000 predictions from ['efficientnet_b4', 'convnext_tiny', 'swin_t'] to /kaggle/working/predictions.csv


,image_id,label,confidence
0,367711e7-e4c2-4a2e-b6ab-1f9d2726fbcd,0,0.072130
1,446427b0-a2b9-4fbc-98b8-b4b3f83fb08a,0,0.073321
2,c1bb4ef6-1107-465b-83aa-8459e6f0d22f,1,0.663355
3,71e01d0a-c2bc-4503-9441-e02f3d42074f,1,0.902443
4,6f88ba6b-8443-49b3-82a7-127e1d9a564a,0,0.071147
